In [43]:
import pandas as pd

df = pd.read_csv("http://114.207.245.181:13000/csv/Rossmann_Store_Sales.csv", low_memory=False)
df

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday
0,1,5,2015-07-31,5263,555,1,1,0,1
1,2,5,2015-07-31,6064,625,1,1,0,1
2,3,5,2015-07-31,8314,821,1,1,0,1
3,4,5,2015-07-31,13995,1498,1,1,0,1
4,5,5,2015-07-31,4822,559,1,1,0,1
...,...,...,...,...,...,...,...,...,...
1017204,1111,2,2013-01-01,0,0,0,0,a,1
1017205,1112,2,2013-01-01,0,0,0,0,a,1
1017206,1113,2,2013-01-01,0,0,0,0,a,1
1017207,1114,2,2013-01-01,0,0,0,0,a,1


In [44]:
df.columns

Index(['Store', 'DayOfWeek', 'Date', 'Sales', 'Customers', 'Open', 'Promo',
       'StateHoliday', 'SchoolHoliday'],
      dtype='object')

In [45]:
t1 = {'Store':'매장', 'DayOfWeek':'요일', 'Date':'날짜',
      'Sales':'매출', 'Customers':'고객수', 'Open':'영업유무',
      'Promo':'프로모션', 'StateHoliday':'공휴일', 'SchoolHoliday':'학교방학'}

df = df.rename(columns=t1)
df.columns

Index(['매장', '요일', '날짜', '매출', '고객수', '영업유무', '프로모션', '공휴일', '학교방학'], dtype='object')

In [46]:
# a => 일반 공휴일
# b => 부활절 휴일
# c => 성탄절
df['공휴일'].value_counts()

공휴일
0    986159
a     20260
b      6690
c      4100
Name: count, dtype: int64

In [47]:
df.dtypes

매장       int64
요일       int64
날짜      object
매출       int64
고객수      int64
영업유무     int64
프로모션     int64
공휴일     object
학교방학     int64
dtype: object

In [48]:
df['날짜'] = pd.to_datetime(df['날짜'])
df['년'] = df['날짜'].dt.year
df['월'] = df['날짜'].dt.month
df['일'] = df['날짜'].dt.day

df.dtypes

매장               int64
요일               int64
날짜      datetime64[ns]
매출               int64
고객수              int64
영업유무             int64
프로모션             int64
공휴일             object
학교방학             int64
년                int32
월                int32
일                int32
dtype: object

In [49]:
# 공휴일 one hot 으로 변경
df1 = pd.get_dummies(
    df,
    columns=['공휴일'],
    drop_first=True,
    dtype=int
)

df1.head(1)

,매장,요일,날짜,매출,고객수,영업유무,프로모션,학교방학,년,월,일,공휴일_a,공휴일_b,공휴일_c
0,1,5,2015-07-31,5263,555,1,1,1,2015,7,31,0,0,0


In [50]:
# 참고용 => 분포변경
import numpy as np
print("원본")
s = np.array([0, 100, 1000, 10000, 100000])
print(s)

print("분포변경")
g = np.log1p(s)     # log(1+x)
print(g)

print("원본")
r = np.expm1(g)
print(r)

원본
[     0    100   1000  10000 100000]
분포변경
[ 0.          4.61512052  6.90875478  9.21044037 11.51293546]
원본
[0.e+00 1.e+02 1.e+03 1.e+04 1.e+05]


In [51]:
y = np.log1p(df['매출']).values
x = df1.drop(columns=['날짜', '매출']).values
x.shape, y.shape

((1017209, 12), (1017209,))

In [52]:
# 데이터 분할
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=123)
x_train.shape, x_test.shape, y_train.shape, y_test.shape

((813767, 12), (203442, 12), (813767,), (203442,))

In [53]:
# 스케일러
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

x_train_scaled.shape, x_test_scaled.shape

((813767, 12), (203442, 12))

In [56]:
from xgboost import XGBRegressor
model = XGBRegressor(random_state=123)
model.fit(x_train_scaled, y_train)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [58]:
from sklearn.metrics import r2_score

y_pred = model.predict(x_test_scaled)
r2_score(y_test, y_pred)

0.9980789401062028

In [62]:
# 모델 생성, 컴파일, 훈련하기
from tensorflow import keras
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization

model = keras.Sequential([
    # 입력층 & 첫 번째 은닉층: 데이터 특성(Feature) 개수를 자동으로 감지하도록 input_shape 설정
    Dense(128, activation="relu", input_shape=(x_train_scaled.shape[1],)),
    BatchNormalization(), # 학습을 안정화하고 속도를 높임
    Dropout(0.2),         # 과적합 방지

    # 두 번째 은닉층
    Dense(64, activation="relu"),
    BatchNormalization(),
    Dropout(0.2),

    # 세 번째 은닉층
    Dense(32, activation="relu"),
    
    # 출력층: 매출 예측(연속형 수치 회귀)이므로 뉴런 수는 1개, 활성화 함수는 없음
    Dense(1, activation="linear")
])

model.summary()

c:\Users\pknukdt\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_8 (Dense)                 │ (None, 128)            │         1,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 12,801 (50.00 KB)

 Trainable params: 12,417 (48.50 KB)

 Non-trainable params: 384 (1.50 KB)

In [64]:
model.compile(optimizer="adam", loss="mse", metrics=["mae"])
model.fit(x_train_scaled, y_train, epochs=500, validation_data=(x_test_scaled, y_test), batch_size=256)

Epoch 1/500
3179/3179 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - loss: 0.0340 - mae: 0.1345 - val_loss: 0.0339 - val_mae: 0.1281
Epoch 2/500
3179/3179 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step - loss: 0.0338 - mae: 0.1337 - val_loss: 0.0330 - val_mae: 0.1328
Epoch 3/500
3179/3179 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step - loss: 0.0336 - mae: 0.1333 - val_loss: 0.0309 - val_mae: 0.1275
Epoch 4/500
3179/3179 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step - loss: 0.0335 - mae: 0.1331 - val_loss: 0.0339 - val_mae: 0.1294
Epoch 5/500
3179/3179 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - loss: 0.0334 - mae: 0.1328 - val_loss: 0.0319 - val_mae: 0.1284
Epoch 6/500
3179/3179 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - loss: 0.0331 - mae: 0.1325 - val_loss: 0.0468 - val_mae: 0.1301
Epoch 7/500
3179/3179 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - loss: 0.0329 - mae: 0.1324 - val_loss: 0.0306 - val_mae: 0.1285
Epoch 8/500
3179/3179 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - loss: 0.0330 - mae: 0.1323 - val_loss: 0.0331 - val_mae: 0.1331
Epoch 9/500
3179/3179 ━━━━━━━━━━

In [65]:
# 예측하기
from sklearn.metrics import r2_score

y_pred_log = model.predict(x_test_scaled)

y_pred = np.expm1(y_pred_log.flatten())
y_test = np.expm1(y_test)

r2_score(y_test, y_pred)

6358/6358 ━━━━━━━━━━━━━━━━━━━━ 3s 517us/step


0.8682365999921466